# Companion demo: the keystone thesis on AITA

This notebook is a reproducible companion to *ErisML and DEME: A Structure-Preserving, Framework-Pluralist Pipeline for Auditable Machine Ethics*. It demonstrates the paper's thesis on **240 real moral dilemmas** from the *Am I the Asshole?* (AITA) corpus, using the **precomputed DEME evaluations** shipped alongside (`aita_deme_results.jsonl`). No LLM, GPU, or network access is required — only `numpy`, `scipy`, `matplotlib`.

Each post was evaluated under **two declared stakeholder ethos** (`dear_abby`, a harm/care advocate; `aita`, a fairness/fidelity community), producing for each: a multi-dimensional **moral vector**, four **framework projections** (consequentialist, Kantian, virtue, care), and a **verdict**. We use these to demonstrate four claims of the paper.

1. **Scalar collapse hides structure** (§2): one number cannot carry what a moral situation contains.
2. **Multi-dimensionality separates situation-harm from culpability** (§5): a harm score does not predict blame; the fairness and care dimensions do.
3. **Declared pluralism surfaces disagreement** (§4–§5): different ethos systematically disagree.
4. **Honest governance** (§5): a worst-off/escalate policy routes genuine disagreement to human review rather than averaging it away.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

ROWS = [json.loads(l) for l in open('aita_deme_results.jsonl', encoding='utf-8')]
# the moderation 10-module evaluator (a task instrument; see paper Appendix A)
DIMS = ['physical_harm', 'rights_respect', 'fairness_equity', 'autonomy_consent',
        'legitimacy_trust', 'epistemic_quality', 'care_protection', 'vow_fidelity',
        'third_party_externality', 'repair_residue']
ETHOS = ['dear_abby', 'aita']
OK = [r for r in ROWS if all(r['per_stakeholder'][e]['ok'] for e in ETHOS)]
# author-at-fault from the gold AITA label: YTA/ESH = culpable, NTA/NAH = not
CULP = {'YTA': 1, 'ESH': 1, 'NTA': 0, 'NAH': 0}
print(f'{len(ROWS)} posts, {len(OK)} evaluable under both ethos')
print('gold-label counts:', {l: sum(r["verdict_label"] == l for r in OK) for l in CULP})

## Thesis 1 — a scalar throws the structure away

Take a single post on which the two ethos **disagree**. A toxicity score or a single safety probability would return one number. DEME instead returns, per ethos, a ten-dimensional moral vector and four framework verdicts — and those frameworks themselves disagree. None of that survives a scalar.

In [ ]:
ex = next(r for r in OK
          if r['per_stakeholder']['dear_abby']['verdict'] != r['per_stakeholder']['aita']['verdict'])
print('post id:', ex['id'], '| gold AITA label:', ex['verdict_label'],
      '| governance action:', ex['agg']['action'])
for e in ETHOS:
    ps = ex['per_stakeholder'][e]
    pol = {k: v['polarity'] for k, v in ps['projections'].items()}
    print(f"\n[{e}] verdict={ps['verdict']}  framework polarities:")
    for k, v in pol.items():
        print(f'    {k:28s} {v}')

x = np.arange(len(DIMS)); w = 0.4
fig, ax = plt.subplots(figsize=(11, 4))
for i, e in enumerate(ETHOS):
    mv = ex['per_stakeholder'][e]['moral_vector']
    ax.bar(x + (i - 0.5) * w, [mv[d] for d in DIMS], w, label=e)
ax.set_xticks(x); ax.set_xticklabels(DIMS, rotation=45, ha='right')
ax.axhline(0, color='k', lw=0.6); ax.set_ylabel('dimension value')
ax.set_title(f"One post, two ethos: the moral structure a scalar would discard (id {ex['id']})")
ax.legend(); plt.tight_layout(); plt.show()

## Thesis 2 — harm ≠ culpability; the multi-dimensional vector separates them

The sharpest evidence that one number is not enough: **aggregate harm is essentially uncorrelated with whether the author is at fault**, while the **fairness** and **care** dimensions track fault. A scalar harm score would discard exactly the dimensions that carry the moral signal.

In [ ]:
culp = np.array([CULP[r['verdict_label']] for r in OK])
prim = [r['per_stakeholder']['dear_abby']['moral_vector'] for r in OK]  # primary ethos
rhos = {}
for d in DIMS:
    rho, p = spearmanr([mv[d] for mv in prim], culp)
    rhos[d] = (rho, p)
order = sorted(DIMS, key=lambda d: rhos[d][0])
colors = ['tab:red' if rhos[d][1] < 0.05 else 'lightgray' for d in order]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(order, [rhos[d][0] for d in order], color=colors)
ax.axvline(0, color='k', lw=0.6)
ax.set_xlabel('Spearman ρ with author culpability  (red = p < 0.05)')
ax.set_title('Which moral dimensions predict blame?')
plt.tight_layout(); plt.show()
for d in ['physical_harm', 'fairness_equity', 'care_protection']:
    print(f'  {d:18s} ρ={rhos[d][0]:+.3f}  p={rhos[d][1]:.3f}')

`physical_harm` is uncorrelated with culpability (n.s.), while `fairness_equity` and `care_protection` are significant — reproducing the paper's §5 finding. (Culpability here is the binary YTA/ESH author-at-fault label; the exact ρ differ slightly from the paper's ordinal coding but the pattern is identical.)

## Probe — how many dimensions are actually doing work?

A down-payment on the claim that moral evaluation is not one-dimensional, and on validating the *moral core* of Appendix A. PCA on the moral vectors shows the **best single scalar captures only ~35% of the moral variance**; five components are needed for 90%. The representation is empirically far from a single number.

In [ ]:
M = np.array([[r['per_stakeholder']['dear_abby']['moral_vector'][d] for d in DIMS] for r in OK])
Mc = M - M.mean(0)
sv = np.linalg.svd(Mc, compute_uv=False)
evr = sv**2 / (sv**2).sum(); cum = np.cumsum(evr)
k90 = int(np.argmax(cum >= 0.90) + 1)
print(f'PC1 (best single scalar) explains {evr[0]:.1%} of moral variance; '
      f'{k90}/{len(DIMS)} components reach 90%')
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(range(1, len(evr) + 1), evr, color='tab:purple', alpha=0.8, label='per component')
ax.plot(range(1, len(cum) + 1), cum, 'k.-', label='cumulative')
ax.axhline(0.9, color='gray', ls=':')
ax.set_xlabel('principal component'); ax.set_ylabel('variance explained')
ax.set_title('A single scalar captures only %.0f%% of the moral variance' % (evr[0] * 100))
ax.legend(); plt.tight_layout(); plt.show()

## Thesis 3 — declared ethos systematically disagree

Pluralism is not noise to average away. The two declared ethos reach **different verdicts on about a third of posts**, with a stable mean separation between their moral vectors.

In [ ]:
da = [r['per_stakeholder']['dear_abby']['verdict'] for r in OK]
ai = [r['per_stakeholder']['aita']['verdict'] for r in OK]
div = np.mean([a != b for a, b in zip(da, ai)])
l2 = np.array([np.linalg.norm(
        np.array([r['per_stakeholder']['dear_abby']['moral_vector'][d] for d in DIMS]) -
        np.array([r['per_stakeholder']['aita']['moral_vector'][d] for d in DIMS])) for r in OK])
print(f'verdict divergence: {div:.1%}    mean L2 between ethos vectors: {l2.mean():.3f}')
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(l2, bins=30, color='tab:blue', alpha=0.8)
ax.axvline(l2.mean(), color='k', ls='--', label=f'mean {l2.mean():.3f}')
ax.set_xlabel('L2 distance between the two ethos moral vectors')
ax.set_ylabel('posts'); ax.set_title('Same situation, different declared values → different evaluation')
ax.legend(); plt.tight_layout(); plt.show()

## Thesis 4 — honest governance: surface disagreement, escalate it

Rather than collapse conflicting frameworks into a verdict, the worst-off/escalate policy **routes genuine disagreement to human review**. Nearly every post exhibits at least two distinct framework polarities, and the governed action is dominated by escalation — the framework's contribution is to surface conflict faithfully, not to adjudicate it.

In [ ]:
def n_polarities(r):
    pol = set()
    for e in ETHOS:
        pol |= {v['polarity'] for v in r['per_stakeholder'][e]['projections'].values()}
    return len(pol)

multi = np.mean([n_polarities(r) >= 2 for r in OK])
esc = np.mean([r['agg'].get('escalated', False) for r in OK])
acts = {}
for r in OK:
    a = r['agg'].get('action'); acts[a] = acts.get(a, 0) + 1
print(f'posts with ≥2 distinct framework polarities: {multi:.1%}')
print(f'escalated to human review: {esc:.1%}')
fig, ax = plt.subplots(figsize=(7, 3.2))
k = sorted(acts, key=acts.get, reverse=True)
ax.bar(k, [acts[a] for a in k], color='tab:green')
ax.set_ylabel('posts'); ax.set_title('Governed action under worst-off / escalate')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()
print('\nConcrete escalated disagreement (id', ex['id'], '):',
      'dear_abby =', ex['per_stakeholder']['dear_abby']['verdict'], '|',
      'aita =', ex['per_stakeholder']['aita']['verdict'], '→', ex['agg']['action'])

## The escalation frontier — tuning integrity vs throughput

Reviewer point: worst-off/escalate sends ~91% of cases to human review, which makes the system a triage tool. But that policy escalates *any* disagreement — including mere differences of degree. If instead we escalate only **decision-relevant** conflicts (some framework says *permit* while another says *forbid*) and grade the rest by a conflict-severity score, we can trace the integrity↔throughput frontier. The result: **escalation can be roughly halved (100% → ~48%) at zero loss of genuine permit-vs-forbid conflicts**; below ~48% lies a real tradeoff.

In [ ]:
RANK = {'permit': 0.0, 'neutral': 0.5, 'escalate': 0.6, 'forbid': 1.0}
def conflict(r):
    pol = [v['polarity'] for e in ETHOS for v in r['per_stakeholder'][e]['projections'].values()]
    span = max(RANK[p] for p in pol) - min(RANK[p] for p in pol)
    dissent = 1 - max(pol.count(p) for p in set(pol)) / len(pol)
    da = np.array([r['per_stakeholder']['dear_abby']['moral_vector'][d] for d in DIMS])
    ai = np.array([r['per_stakeholder']['aita']['moral_vector'][d] for d in DIMS])
    hard = ('permit' in pol) and ('forbid' in pol)  # decision-relevant: permit vs forbid
    return span, dissent, np.linalg.norm(da - ai), hard

C = np.array([conflict(r)[:3] for r in OK]); hard = np.array([conflict(r)[3] for r in OK])
severity = 0.6 * C[:, 0] + 0.25 * C[:, 1] + 0.15 * C[:, 2] / C[:, 2].max()
taus = np.linspace(0, severity.max(), 80)
esc = np.array([(severity >= t).mean() for t in taus])
integ = np.array([(severity[hard] >= t).mean() for t in taus])
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(esc * 100, integ * 100, '-', color='tab:red', lw=2)
for label, t in [('worst-off (escalate any)', 0.0),
                 ('decision-relevant only', severity[hard].min()),
                 ('30% budget', np.quantile(severity, 0.70))]:
    x, y = (severity >= t).mean() * 100, (severity[hard] >= t).mean() * 100
    ax.scatter([x], [y], zorder=5)
    ax.annotate(label, (x, y), textcoords='offset points', xytext=(6, -6), fontsize=8)
ax.set_xlabel('escalation rate (% to human review)  →  lower = cheaper')
ax.set_ylabel('decision-relevant conflicts preserved (%)')
ax.set_title('Integrity ↔ throughput frontier (tunable escalation)')
plt.tight_layout(); plt.show()
t0 = severity[hard].min()
print(f'decision-relevant permit-vs-forbid conflicts: {hard.mean():.1%} of posts')
print(f'escalate-decision-relevant-only: {(severity>=t0).mean():.0%} escalation at 100% integrity '
      f'(vs ~91% under worst-off) — about half the review load, no genuine conflict lost')

## Conclusion

On 240 real moral dilemmas, the precomputed DEME evaluations reproduce the paper's claims directly: a single scalar would discard the moral structure (Thesis 1); aggregate harm does not predict blame while fairness and care do (Thesis 2); declared ethos disagree on ~34% of cases (Thesis 3); and the worst-off/escalate policy surfaces that disagreement and routes it to human review rather than averaging it away (Thesis 4). All figures here are computed from `aita_deme_results.jsonl` with only `numpy`/`scipy`/`matplotlib` — the reproducibility artifact for the paper's §5.